# Synthetic Data Creation

This notebook documents how the thesis dataset is generated from scratch. It shows the planning-phase variables, the Monte Carlo generation logic, the structural stratification, the injection of missing data and outliers, and the creation of the final model-ready synthetic dataset.

## Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")

## Paths and configuration

In [ ]:
if os.path.basename(os.getcwd()) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
else:
    PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ExperimentConfig
from src.data_generation import (
    add_engineered_features,
    generate_synthetic_projects,
    inject_mcar_missingness,
    inject_predictor_outliers,
)
from src.utils import build_summary_statistics, build_variable_dictionary, set_global_seed

CONFIG_PATH = os.path.join(PROJECT_ROOT, "outputs", "tables", "run_config.json")
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as handle:
        config_dict = json.load(handle)
    cfg = ExperimentConfig(**config_dict)
else:
    cfg = ExperimentConfig()

set_global_seed(cfg.random_state, cfg.python_hash_seed)
print(f"Project root: {PROJECT_ROOT}")
print(f"n_projects: {cfg.n_projects}")
print(f"missing_rate: {cfg.missing_rate}")
print(f"outlier_rate: {cfg.outlier_rate}")

## Generate the synthetic datasets

In [ ]:
# Generate each stage explicitly so the workflow is visible to examiners.
raw_df = generate_synthetic_projects(cfg)
outlier_df = inject_predictor_outliers(raw_df, cfg)
missing_df = inject_mcar_missingness(outlier_df, cfg)
model_df = add_engineered_features(missing_df)

print("Raw dataset:", raw_df.shape)
print("After outlier injection:", outlier_df.shape)
print("After missingness injection:", missing_df.shape)
print("Model-ready dataset:", model_df.shape)
display(model_df.head())

## Structural stratification check

In [ ]:
cross_tab = pd.crosstab(model_df["project_scale_tier"], model_df["project_complexity_tier"])
display(cross_tab)

plt.figure(figsize=(8, 6))
sns.heatmap(cross_tab.div(cross_tab.sum(axis=1), axis=0), annot=True, fmt=".2f", cmap="Blues")
plt.title("Complexity mix within each project scale tier")
plt.xlabel("Complexity tier")
plt.ylabel("Scale tier")
plt.tight_layout()
plt.show()

## Feature and target distributions

In [ ]:
selected_cols = [
    "planned_budget",
    "planned_duration",
    "financial_uncertainty",
    "planning_maturity_score",
    "cost_overrun_pct",
    "schedule_overrun_pct",
]

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.ravel()
for ax, col in zip(axes, selected_cols):
    sns.histplot(model_df[col], kde=True, bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Missing data and outlier inspection

In [ ]:
missing_rates = missing_df.isna().mean().sort_values(ascending=False)
display(missing_rates[missing_rates > 0])

compare_cols = ["planned_budget", "planned_duration", "financial_uncertainty", "technical_uncertainty"]
quantile_rows = []
for col in compare_cols:
    quantile_rows.append(
        {
            "variable": col,
            "raw_q99": raw_df[col].quantile(0.99),
            "outlier_q99": outlier_df[col].quantile(0.99),
            "raw_max": raw_df[col].max(),
            "outlier_max": outlier_df[col].max(),
        }
    )
quantile_df = pd.DataFrame(quantile_rows)
display(quantile_df)

## Variable dictionary and summary statistics

In [ ]:
variable_dictionary_df = build_variable_dictionary(cfg.include_optional_planning_features)
summary_statistics_df = build_summary_statistics(model_df)

print("Variable dictionary")
display(variable_dictionary_df)
print("Summary statistics")
display(summary_statistics_df.head(20))

## Short summary

In [ ]:
print("This notebook documents the synthetic data creation process used in the thesis.")
print("It shows the structural tiers, the planning-stage variables, the injected missingness and outliers, and the engineered model-ready dataset.")
print("For examiners, it makes the data-generation logic inspectable without reading the source files directly.")